# 3 - Collect results and analyze domain shift

In [ ]:
import digitalhub as dh

In [ ]:
project = dh.get_or_create_project("cflv-1")

## Log inputs

In [ ]:
artifact_names = {
    "training_results": "2-abmil-conch-v15-results",
    "internal_features": "1-trident-features-internal-test",
    "external_features": "1-trident-features-external-test",
}
local_paths = {
    "training_results": "runs",
    "internal_features": "/data/features/internal_test",
    "external_features": "/data/features/external_test",
}

inputs = {}
for name, artifact_name in artifact_names.items():
    inputs[name] = project.get_artifact(artifact_name)
print(name, inputs[name].key)

## Declaration

In [ ]:
collection_function = project.new_function(
    name="3-collect-results", kind="python", python_version="PYTHON3_11",
    requirements="3-analysis/requirements.txt", code_src="3-analysis",
    handler="main:collect_results",
    # image=collection_image_ref
)

In [ ]:
collection_function = project.get_function("3-collect-results",
    entity_id="494f101a164e4980a0ccf731034150a2"
)

In [ ]:
shift_function = project.new_function(
    name="3-analyze-shift", kind="python", python_version="PYTHON3_11",
    requirements="3-analysis/requirements.txt", code_src="3-analysis",
    handler="main:analyze_shift",
    # image=shift_image_ref
)

In [ ]:
shift_function = project.get_function("3-analyze-shift",
    entity_id="7d49da2a36ee49d69e855f1d55aa6f58"
)

## Building

In [ ]:
build_options = {
    "instructions": [
        "apt-get update && apt-get install -y git libopenslide0 libgl1 libglib2.0-0 && rm -rf /var/lib/apt/lists/*"
    ],
    "resources": {"cpu": "8", "mem": "16Gi", "disk": "20Gi"},
}

In [ ]:
collection_build = collection_function.run(action="build", **build_options)

In [ ]:
collection_image_ref = collection_function.refresh().spec.image

In [ ]:
shift_build = shift_function.run(action="build", **build_options)

In [ ]:
shift_image_ref = shift_function.refresh().spec.image

## Running

In [ ]:
collection_run = collection_function.run(action="job",
    inputs={"training_results": inputs["training_results"].key},
    parameters={"output_artifact_name": "3-collection", "top_k": 10},
    resources={"cpu": "2", "mem": "8Gi", "disk": "20Gi"},
    local_execution=False,
)
print(collection_run.key)

## 3.1 - Shared shift settings

`exclude_external_ssl=True` preserves the study's exclusion of external slide names matching its pure sessile-serrated pattern. Set it False to include all external slides. With False, upstream columns still named `external_no_ssl` refer to the included external set; `digitalhub_inputs.json` records the setting. Top-64 selects highest-attention patches, not explicit positive-class contributions.

In [ ]:
dh.get_k8s_resource_profiles()

In [ ]:
gpu_profile = '1xl40s'  # Set an available one-GPU profile if required.
shift_resources = {"cpu": "8", "mem": "32Gi", "gpu": "1", "disk": "20Gi"}
shift_inputs = {name: artifact.key for name, artifact in inputs.items()}
shift_parameters = {
    "feature_subdirs": ["20x_512px_0px_overlap/features_conch_v15"],
    "exclude_external_ssl": True,
    "n_repeats": 20,
    "target_per_domain": 80,
    "sample_external": True,
    "seed": 1234,
    "knn_k": 15,
    "sigma_sample_size": 512,
    "energy_sample_size": 2048,
    "energy_chunk_size": 512,
    "eps": 1e-12,
    "device": "cuda",
    "batch_size": None,  # Layer extraction defaults to training config's eval_batch_size.
}
# For CPU runs: set device='cpu', remove 'gpu' from shift_resources, and use profile=None.

## 3.2 - Foundation-model feature shift

In [ ]:
fm_shift_run = shift_function.run(action="job",
    inputs=shift_inputs,
    parameters={**shift_parameters, "analysis": "fm", "output_artifact_name": "3-fm-shift"},
    profile=gpu_profile, resources=shift_resources, local_execution=False,
)

## 3.3 - Default ABMIL layers

In [ ]:
layer_shift_run = shift_function.run(action="job",
    inputs=shift_inputs,
    parameters={**shift_parameters, "analysis": "layers", "output_artifact_name": "3-abmil-layer-shift"},
    profile=gpu_profile, resources=shift_resources, local_execution=False,
)

## 3.4 - Top-64 analysis

In [ ]:
top64_shift_run = shift_function.run(action="job",
    inputs=shift_inputs,
    parameters={**shift_parameters, "analysis": "top64", "output_artifact_name": "3-abmil-top64-shift"},
    profile=gpu_profile, resources=shift_resources, local_execution=False,
)